# 🧠 Scouting Report Agent: RAG Pipeline with Databricks + LangChain

This notebook builds and deploys a Retrieval-Augmented Generation (RAG) agent using:

- **LangChain** for prompt handling and agent logic
- **Databricks Vector Search** to retrieve relevant scouting reports
- **MLflow** to package and deploy the agent
- **Databricks Model Serving** and **Review App** for interactive feedback

---

## 👇 What this notebook covers:

1. 🛠 Install required dependencies  
2. ⚙️ Configure your catalog, schema, model, and index  
3. 🔍 Set up prompt + retrieval logic with player-aware context  
4. 🤖 Define an agentic chain that tracks topics and handles comparisons  
5. 📦 Log and deploy the agent using MLflow  
6. ✅ Test queries and launch the Review App


# 📦 Install LangChain + Databricks SDKs

In [0]:
# Core LangChain and Databricks integrations
%pip install --upgrade --force-reinstall \
  databricks-sdk==0.50.0 \
  databricks-vectorsearch \
  databricks-langchain \
  langchain \
  langchain-community \
  databricks-agents

# Pin compatible PyData stack versions (match DBR 14+)
%pip install --upgrade --force-reinstall \
  numpy==1.26.4 \
  pandas==1.5.3 \
  pyarrow==12.0.1

# Restart Python
dbutils.library.restartPython()

INFO: pip is looking at multiple versions of databricks-agents to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/692.3 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 692.3/692.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.5 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 72.2 MB/s eta 0:00:00

*** WARNING: max output size exceeded, skipping output. ***

l: nest-asyncio
    Found existing installation: nest-asyncio 1.6.0
    Uninstalling nest-asyncio-1.6.0:
      Successfully uninstalled nest-asyncio-1.6.0
  Attempting uninstall: mypy-extens

# 🤖 Import Core Libraries

In [0]:
import re
import mlflow
from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate

# ⚙️ Set Up Configuration and Agentic Chain

In [0]:
# Define a custom LLM flow that tracks history, detects player changes, and routes retrieval intelligently.”

# Autolog metrics and traces
mlflow.langchain.autolog()

# Convention when logging a LangChain model with MLflow
def loader_fn(_context=None):
    # --- Configuration ---
    CATALOG = "alexander_booth"
    SCHEMA = "rag_demo"
    INDEX_NAME = "scouting_reports_index"
    LLM_MODEL = "databricks-llama-4-maverick"

    # --- Core Components ---
    # LLM endpoint (chat-compatible)
    llm = ChatDatabricks(endpoint=LLM_MODEL)

    # Vector search index — built from scouting reports
    vs = DatabricksVectorSearch(
        index_name=f"{CATALOG}.{SCHEMA}.{INDEX_NAME}",
        columns=["primary_key", "combined_text", "playerName", "Season", "ID", "minorMasterId"]
    )
    retriever = vs.as_retriever(search_kwargs={"k": 1})  # k=1: one document per player

    # --- Prompt Template ---
    # Defines how context, chat history, and the question are injected into the LLM
    prompt_template = PromptTemplate(
        input_variables=["context", "question", "chat_history"],
        template="""
        You are a baseball scouting assistant. Use the following scouting reports to answer the question. If the user asks for a comparison, include strengths/weaknesses for each player.

        Context:
        {context}

        Chat History:
        {chat_history}

        Question: {question}
        Answer in full sentences, using only the context provided.
        """
    )

    # --- Player Name Extractor ---
    # Uses a simple regex to identify full names (e.g., "Dylan Crews") in user questions
    def extract_all_players(text):
        return re.findall(r"\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b", text)

    # --- Main Chain Function ---
    def run_chain(inputs):
        # --- Step 1: Normalize Input Format ---
        # LangChain-style message list
        if isinstance(inputs, list) and all(hasattr(m, "content") for m in inputs):
            messages = [{"role": "user" if isinstance(m, HumanMessage) else "assistant", "content": m.content} for m in inputs[:-1]]
            query = inputs[-1].content
        # Dict from API request (with keys: 'query', 'messages')
        elif isinstance(inputs, dict):
            query = inputs.get("query") or inputs.get("question", "")
            messages = inputs.get("messages", [])
        # Plain string input
        elif isinstance(inputs, str):
            query = inputs
            messages = []
        else:
            raise ValueError("Unsupported input format")

        # --- Step 2: Extract Player Names ---
        # Detects if the user is asking about multiple specific players
        player_names = extract_all_players(query)

        # --- Step 3: Retrieve Scouting Reports (with player switch detection) ---
        # Extract player(s) from first user message (if exists)
        prev_user_msgs = [m["content"] for m in messages if m["role"] == "user"]
        first_msg = prev_user_msgs[0] if prev_user_msgs else None
        initial_players = extract_all_players(first_msg) if first_msg else []

        # Extract player(s) from current query
        current_players = extract_all_players(query)

        # Case 1: Multi-player query → retrieve all players mentioned
        if len(current_players) > 1:
            docs = []
            for name in current_players:
                docs.extend(retriever.get_relevant_documents(name))

        # Case 2: One player mentioned → compare to initial player (if any)
        elif len(current_players) == 1:
            current = current_players[0]
            if initial_players and current != initial_players[0]:
                # Player has changed → reset context
                docs = retriever.get_relevant_documents(current)
                messages = []  # Clear chat history for prompt
            else:
                # Same player → include first + current message
                retrieval_query = "\n".join([first_msg, query]) if first_msg else query
                docs = retriever.get_relevant_documents(retrieval_query)

        # Case 3: No player name mentioned → fallback to semantic context
        else:
            retrieval_query = "\n".join(prev_user_msgs + [query]) if prev_user_msgs else query
            docs = retriever.get_relevant_documents(retrieval_query)

        # --- Step 4: Format Retrieved Context for Prompt ---
        # Clearly labels each retrieved report with the player's name
        context = "\n\n".join(
            f"{doc.metadata.get('playerName', 'Unknown')} report:\n{doc.page_content}"
            for doc in docs
        )

        # --- Step 5: Build Chat History for Prompt ---
        # Preserves conversation flow in prompt (LLM context), but doesn't influence retrieval
        chat_history = "\n".join(f"{m['role']}: {m['content']}" for m in messages)

        # --- Step 6: Build and Run Prompt ---
        prompt = prompt_template.format(
            question=query,
            chat_history=chat_history,
            context=context
        )
        response = llm.invoke(prompt)
        answer = response.content if hasattr(response, "content") else response

        # --- Step 7: Format Source Metadata for Output ---
        # Includes player, season, and report IDs so the response is traceable
        sources_md = "\n\n".join(
            f"**Player:** `{doc.metadata.get('playerName', 'Unknown')}`\n"
            f"**Season:** `{str(int(doc.metadata['Season'])) if 'Season' in doc.metadata else 'N/A'}`\n"
            f"**Fangraphs ID:** `{doc.metadata.get('minorMasterId', 'Unknown')}`\n"
            f"**Report ID:** `{doc.metadata.get('primary_key', 'N/A')}`"
            for doc in docs
        )

        # --- Final Markdown-formatted Response ---
        return f"### 🧠 Answer\n{answer}\n\n---\n### 📚 Sources\n{sources_md}"

    return RunnableLambda(run_chain)


# 🧠 Finalize Environment
# Log Run to MLflow Experiment

In [0]:
# Check Core Library Versions
!pip freeze | grep -E 'mlflow|numpy|pandas|pyarrow|langchain|databricks'

databricks-agents==0.21.0
databricks-ai-bridge==0.5.1
databricks-connect==16.1.6
databricks-langchain==0.5.1
databricks-sdk==0.50.0
databricks-vectorsearch==0.56
ipywidgets @ file:///databricks/.virtualenv-def/ipywidgets-7.7.2-2databricksnojsdeps-py2.py3-none-any.whl#sha256=903ead20c8d40de671853515fcad2f34b43ebf3eff80e4df3f876b8dd64c903b
langchain==0.3.25
langchain-community==0.3.25
langchain-core==0.3.65
langchain-text-splitters==0.3.8
mlflow==2.22.1
mlflow-skinny==2.22.1
numpy==1.26.4
pandas==1.5.3
pyarrow==12.0.1
unitycatalog-langchain==0.2.0


In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint

# --- Configuration ---
CATALOG = "alexander_booth"
SCHEMA = "rag_demo"
INDEX_NAME = "scouting_reports_index"
LLM_MODEL = "databricks-llama-4-maverick"

# --- Example input ---
example_input = {
    "query": "Tell me about Jace LaViolette.",
    "messages": []
}

# --- Log the model into MLFlow ---
with mlflow.start_run():
    logged_chain_info = mlflow.langchain.log_model(
        lc_model=loader_fn(),       
        artifact_path="abooth_scouting_reports_agent",
        input_example=example_input,
        resources=[
        DatabricksVectorSearchIndex(index_name=f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"),
        DatabricksServingEndpoint(endpoint_name=LLM_MODEL)
        ],
        pip_requirements=[                        # Explicit pip dependencies
            "mlflow==2.22.1",
            "cloudpickle==3.1.1",
            "langchain==0.3.25",
            "pydantic==2.11.7",
            "databricks-vectorsearch==0.56",
            "databricks-sdk==0.50.0",
            "databricks-agents==0.21.0",
            "databricks-langchain==0.5.1",
            "langchain-community==0.3.25",
            "langchain-core==0.3.65",
            "numpy==1.26.4",
            "pandas==1.5.3",
            "pyarrow==12.0.1"
        ]
    )

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

# 🧪 Run Test Queries

In [0]:
# Agent Testing
chain = loader_fn()

# Test with Agent Framework-style input
agent_input = [
    HumanMessage(content="Tell me about Jace LaViolette."),
    AIMessage(content="He’s a left-handed outfielder with power."),
    HumanMessage(content="What are his weaknesses?")
]

print(chain.invoke(agent_input))

print("\n\n")

# Test with direct invocation
example_input = {
    "query": "What are his weaknesses?",
    "messages": [
        {"role": "user", "content": "Tell me about Jace LaViolette."},
        {"role": "assistant", "content": "He’s a left-handed outfielder with power."}
    ]
}

print(chain.invoke(example_input))


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
### 🧠 Answer
Roman Anthony has a few weaknesses. One of his weaknesses is that he can be late on fastballs, particularly those that run up and away from him, although he has become more adept at covering these pitches. Additionally, adjustments to his swing have caused him to concede some of his ability to scoop low pitches, as his cut often has a downward path. He is also a patient hitter who runs deep counts, which has contributed to his strikeout rates hovering in the 23-25% range. Furthermore, his swing isn't totally optimized to get to all of his raw power in games right now, which may take some time to develop.

---
### 📚 Sources
**Player:** `Roman Anthony`
**Season:** `2025`
**Fangraphs ID:** `sa3020211`
**Report ID:** `sa3020211_2025`



[NOTICE] Using a notebook authenti

[Trace(request_id=tr-f2bc4c48d8d2472eb0644be4dd92ec2b), Trace(request_id=tr-c984782b99954398aa83607be5476f17)]

In [0]:
# Test with Player Comparisons
agent_input_raw = [HumanMessage(content="Compare Ethan Holliday to Jace LaViolette")]
print(chain.invoke(agent_input_raw)) 

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
### 🧠 Answer
Sebastian Walcott and Roman Anthony are both highly touted prospects with impressive physical abilities and raw power. While both players have elite bat speed, Walcott's is generated with shockingly little effort for an 18-year-old hitter, whereas Anthony's hands build enormous momentum before exploding with bat speed. Walcott is still a relatively raw prospect with some mechanical flaws in his swing, such as a tendency to pull the ball and chase soft stuff low and away, whereas Anthony has made adjustments to his swing, becoming looser and more active, which h

Trace(request_id=tr-894bb24b91894702ad05f06e39cc6c1e)

# 🚀 Register and Deploy to Databricks Model Serving

In [0]:
from databricks import agents

# Config
MODEL_NAME = "scouting_reports_agent"
MODEL_NAME_FQN = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"

# Register to UC
logged_model_uri = logged_chain_info.model_uri # update as needed
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_model_uri,
    name=MODEL_NAME_FQN
)


Registered model 'alexander_booth.rag_demo.scouting_reports_agent' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

Created version '13' of model 'alexander_booth.rag_demo.scouting_reports_agent'.


In [0]:
# Deploy the agent to Model Serving
deployment_info = agents.deploy(
    model_name=MODEL_NAME_FQN,
    model_version=uc_registered_model_info.version,
    scale_to_zero=True,
    task="retrieval_qa"
)

# 👀 Create and Configure Access to the Review App

In [0]:
# Add Reviewer Instructions
instructions = """
You are reviewing a baseball scouting assistant. Test it by asking about prospects and their strengths or weaknesses.

- Use full player names.
- Try varying question styles (e.g. 'Is he a good hitter?' vs 'What are his strengths?').
- Use the thumbs up/down to rate documents, and suggest better answers if needed.
"""

agents.set_review_instructions(MODEL_NAME_FQN, instructions)

In [0]:
user_list = ["alexander.booth@databricks.com"]

# Set the permissions.
agents.set_permissions(model_name=MODEL_NAME_FQN, users=user_list, permission_level=agents.PermissionLevel.CAN_QUERY)

In [0]:
from databricks.agents import review_app

# Get the review app
my_app = review_app.get_review_app()

# Print the URL of the review app
print(my_app.url)

https://e2-demo-field-eng.cloud.databricks.com/ml/review-v2/6c07a28dbfba43f89da7e441a656ed95


## ✅ Review Your Agent in the Review App

Your RAG agent is now deployed and ready for interactive testing!

In the review app, you can:
- Ask questions about players
- Try vague or follow-up questions like “What are his weaknesses?”
- Give feedback using 👍 / 👎
- Suggest better answers or documents

Once you're satisfied, we'll promote the agent to production via Databricks Apps!
